In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.nem_battery.load import load_raw

df = load_raw("SA1")

Found 60 files


### 1. EDA 

In [11]:
# check nulls 
print(df.isnull().sum())

region             0
settlement_date    0
total_demand_mw    0
rrp                0
period_type        0
interval_start     0
dtype: int64


In [ ]:
# category values
print(df["region"].unique())
print(df["period_type"].value_counts())

<ArrowStringArray>
['SA1']
Length: 1, dtype: str
period_type
TRADE    525888
Name: count, dtype: int64


In [13]:
# price range and negative prices 
print(df["rrp"].describe())

neg = (df["rrp"] < 0)
print(f"Negative intervals: {neg.sum():,} ({neg.mean():.1%} of all intervals)")
print("Lowest price:", df["rrp"].min(), "at", df.loc[df["rrp"].idxmin(), "settlement_date"])
print("Highest price:", df["rrp"].max(), "at", df.loc[df["rrp"].idxmax(), "settlement_date"])

count    525888.000000
mean         98.737607
std         479.997943
min       -1000.000000
25%           0.000000
50%          67.575000
75%         136.040000
max       20300.000000
Name: rrp, dtype: float64
Negative intervals: 129,682 (24.7% of all intervals)
Lowest price: -1000.0 at 2021-10-13 08:45:00
Highest price: 20300.0 at 2026-01-18 19:30:00


In [ ]:
neg = (df["rrp"] < 0)   # True/False for each interval
neg.sum()               # count of True = 129,682
neg.mean()              # share of True = 24.7%


np.float64(0.24659623341852258)

In [14]:
# demand range 
print(df["total_demand_mw"].describe())
print("Zero or negative demand:", (df["total_demand_mw"] <= 0).sum())

count    525888.000000
mean       1295.163786
std         438.277963
min        -311.100000
25%        1106.020000
50%        1354.150000
75%        1544.132500
max        3326.910000
Name: total_demand_mw, dtype: float64
Zero or negative demand: 1222


In [15]:
# rows per month 
per_month = df.groupby(df["interval_start"].dt.to_period("M")).size()
print(per_month.value_counts())

8928    35
8640    20
8064     4
8352     1
Name: count, dtype: int64


Summary: 
- The average price ~ $98.74 and the median $67.58 -> the mean is 46% higher than the median. Rare, huge spikes drag the mean up, so the data is right-skewed.
- Standard deviation $480: about 5x the mean, which means extreme volatility
- Negative prices $24.7 of intervals: about 1 in 4 intervals in SA had a negative price 
- Minium price: -$1000 on 13/10/2021 8:45 market price floor 
- Maximum price: $20300 on 18/10/2026 19:30 market price gap 
- Negative interval: 129,628 intervals = 450 days worth spread over 5 years = 90 days worth per year = 6 hours of every day had negative prices